In [64]:
#run this everytime an edit is made in utils.py so that we can use the helper functions in this Python Notebook
import importlib
import utils
importlib.reload(utils)

<module 'utils' from '/Users/matthewliew/FINM 375-a/matthew-work/hw-2/utils.py'>

In [4]:
import pandas as pd 
import numpy as np

#Loading the rate info
rates_info = pd.read_excel("cap_curves_2025-06-30.xlsx")
display(rates_info.head())

#Loading the swap data
swap_data = pd.read_excel("swaption_vol_data_2025-06-30.xlsx")
display(swap_data.head())

,tenor,swap rates,spot rates,discounts,forwards,flat vols,fwd vols
0,0.25,0.042353,0.042353,0.989523,NaN,NaN,NaN
1,0.50,0.040859,0.040852,0.979883,0.039351,0.156842,0.156842
2,0.75,0.039391,0.039372,0.971043,0.036414,0.180709,0.201708
3,1.00,0.038115,0.038083,0.962807,0.034217,0.204576,0.240464
4,1.25,0.036704,0.036653,0.955417,0.030938,0.242127,0.328341


,reference,instrument,model,date,expiration,tenor,-200,-100,-50,-25,0,25,50,100,200
0,SOFR,swaption,black,2025-06-30,1,1,72.250,46.870,39.100,36.000,33.39,31.300,29.760,28.17,28.660
1,SOFR,swaption,black,2025-06-30,1,2,65.780,44.400,37.970,35.460,33.39,31.750,30.530,29.19,29.300
2,SOFR,swaption,black,2025-06-30,1,3,57.870,40.610,35.560,33.650,32.11,30.920,30.060,29.14,29.290
3,SOFR,swaption,black,2025-06-30,1,4,54.405,38.565,33.925,32.195,30.83,29.805,29.095,28.43,28.885
4,SOFR,swaption,black,2025-06-30,1,5,50.940,36.520,32.290,30.740,29.55,28.690,28.130,27.72,28.480


In [44]:
#Grabbing relavant row
swap_row = swap_data.iloc[3]
swap_row

reference                    SOFR
instrument               swaption
model                       black
date          2025-06-30 00:00:00
expiration                      1
tenor                           4
-200                       54.405
-100                       38.565
-50                        33.925
-25                        32.195
0                           30.83
25                         29.805
50                         29.095
100                         28.43
200                        28.885
Name: 3, dtype: object

## 1.1
Calculate the (relevant) forward swap rate. That is, the one-year forward 4-year swap rate.

In [11]:
one_year_forward_4_yr_swap = utils.find_forward_swap_rate(rate_infos=rates_info, t=0, t_forward=1, maturity=5)
print(round(one_year_forward_4_yr_swap,5))

0.0327


## 1.2
Price the swaptions at the quoted implied volatilites and corresponding strikes, all using the just-calculated forward swap rate as the underlying.

In [ ]:
#Get relavant metrics to calculate the swaption price
offsets = swap_row.index[6:].astype(int).values
offsets_decimal = offsets / 10000
strikes = one_year_forward_4_yr_swap + offsets_decimal
implied_vols = swap_row[offsets].values
discount_rate = utils.compute_swap_annuity(rate_infos=rates_info, t_forward=1, maturity=5)
df = pd.DataFrame({"offsets": offsets, "strikes": strikes, "implied_vol": implied_vols/100, "discount_rate": discount_rate})
df

,offsets,strikes,implied_vol,discount_rate
0,-200,0.012698,0.54405,3.603238
1,-100,0.022698,0.38565,3.603238
2,-50,0.027698,0.33925,3.603238
3,-25,0.030198,0.32195,3.603238
4,0,0.032698,0.3083,3.603238
5,25,0.035198,0.29805,3.603238
6,50,0.037698,0.29095,3.603238
7,100,0.042698,0.2843,3.603238
8,200,0.052698,0.28885,3.603238


In [70]:
F = one_year_forward_4_yr_swap
T = 1.0
N = 100

df["price"] = df.apply(
    lambda row: utils.black_swaption_price(
        F=one_year_forward_4_yr_swap,
        K=row["strikes"],
        sigma=row["implied_vol"],
        T=1.0,
        annuity=discount_rate,
        notional=100,
        payer=True
    ),
    axis=1
)
df

,offsets,strikes,implied_vol,discount_rate,price
0,-200,0.012698,0.54405,3.603238,7.271096
1,-100,0.022698,0.38565,3.603238,3.948049
2,-50,0.027698,0.33925,3.603238,2.536235
3,-25,0.030198,0.32195,3.603238,1.943130
4,0,0.032698,0.3083,3.603238,1.443366
5,25,0.035198,0.29805,3.603238,1.042391
6,50,0.037698,0.29095,3.603238,0.736560
7,100,0.042698,0.2843,3.603238,0.355738
8,200,0.052698,0.28885,3.603238,0.087983


## 1.3
To consider how the expiration and tenor matter, calculate the prices of a few other swaptions for comparison.
- No need to get other implied vol quotes–just use the ATM implied vol you have for the swaption above. (Here we are just interested in how Black’s formula changes with changes in tenor and expiration.)
- No need to calculate for all the strikes–just do the ATM strike.

Alternate swaptions

- The 3mo x 4yr swaption
- The 2yr x 4yr swaption
- the 1yr x 2yr swaption

Report these values and compare them to the price of the 1y x 4y swaption.

In [84]:
sigma = 30.83 / 100

def price_atm_swaption(t_forward, tenor):
    maturity = t_forward + tenor
    
    F = utils.find_forward_swap_rate(rates_info, 0, t_forward, maturity)
    A = utils.compute_swap_annuity(rates_info, t_forward, maturity)
    
    return utils.black_swaption_price(
        F=F,
        K=F,
        sigma=sigma,
        T=t_forward,
        annuity=A,
        notional=100,
        payer=True
    )

price_3m4y = price_atm_swaption(0.25, 4)
price_2y4y = price_atm_swaption(2, 4)
price_1y2y = price_atm_swaption(1, 2)

print(f"3mo x 4yr: {round(price_3m4y, 4)}")
print(f"2yr x 4yr : {round(price_2y4y, 4)}")
print(f"1yr x 2yr : {round(price_1y2y, 4)}")

3mo x 4yr: 0.7485
2yr x 4yr : 2.0599
1yr x 2yr : 0.7108


Using the same ATM implied volatility (30.83%) for all cases, the swaption prices differ solely due to changes in expiry and tenor. At-the-money, Black’s formula implies that price scales approximately with the swap annuity and with the square root of expiry. Holding tenor fixed at 4 years, increasing expiry from 3 months to 1 year to 2 years increases the price because the volatility term grows with √T. Thus, the 2y × 4y swaption is the most expensive among those with 4-year tenor, while the 3m × 4y is the cheapest. Comparing 1y × 4y to 1y × 2y, the shorter 2-year tenor produces a lower price because the swap annuity is smaller, meaning there are fewer discounted fixed payments underlying the option. Overall, longer expiries increase price through greater time uncertainty, and longer tenors increase price through a larger annuity (greater exposure).